# Cleaning US Census Data — Solution Notebook

**Goal:** Turn 10 overlapping state-level census CSV extracts into one analysis-ready data frame.

This notebook contains complete, working code for every task plus alternate implementations, extra practice, and a parameterised simulation section.

![Flowchart](us_census_cleaning_flowchart.png)

## 0. Setup & Libraries

In [ ]:
# Primary tidyverse path
library(readr)
library(dplyr)
library(tidyr)
library(stringr)   # optional but convenient for str_remove

# For plots in the simulation / practice sections
library(ggplot2)

## 1–6. Load and Inspect the Data

**Task:** Discover every `states_*.csv`, read them, bind into one data frame, and inspect.

In [ ]:
# 3. List the files
files <- list.files(path = "data", pattern = "states_.*\\.csv", full.names = TRUE)
print(files)

# 4. Read each into a list of data frames
df_list <- lapply(files, read_csv)

# 5. Concatenate
us_census <- bind_rows(df_list)

# 6. Inspect
cat("\n--- colnames ---\n")
print(colnames(us_census))
cat("\n--- str ---\n")
str(us_census)
cat("\n--- head ---\n")
print(head(us_census))
cat("\nNumber of rows:", nrow(us_census), "\n")

### Alternate (base R)
```r
files <- list.files(path = "data", pattern = "states_.*\\.csv", full.names = TRUE)
df_list <- lapply(files, read.csv, stringsAsFactors = FALSE)
us_census <- do.call(rbind, df_list)
```

## 7. Drop the meaningless index column (X1 / ...1 / Unnamed: 0)

In [ ]:
# The first column is just a row counter from the original CSVs
us_census <- us_census %>% select(-1)   # position-based; robust across naming
# Alternative: us_census <- us_census %>% select(-X1) or select(-`...1`)
head(us_census)

## 8. Remove the % symbol from race columns

In [ ]:
race_cols <- c("Hispanic", "White", "Black", "Native", "Asian", "Pacific")

us_census <- us_census %>%
  mutate(across(all_of(race_cols), ~ gsub("%", "", .x)))

head(us_census[, c("State", race_cols)])

### Alternate (stringr or base loop)
```r
us_census <- us_census %>% mutate(across(all_of(race_cols), ~ str_remove(.x, "%")))
# or
for (col in race_cols) us_census[[col]] <- gsub("%", "", us_census[[col]])
```

## 9. Remove the $ from Income

In [ ]:
us_census <- us_census %>%
  mutate(Income = gsub("\\$", "", Income))   # \\ because $ is regex special

head(us_census[, c("State", "Income")])

## 10–11. Separate GenderPop and strip M / F

In [ ]:
us_census <- us_census %>%
  separate(GenderPop, into = c("male_pop", "female_pop"), sep = "_") %>%
  mutate(
    male_pop   = gsub("M", "", male_pop),
    female_pop = gsub("F", "", female_pop)
  )

head(us_census[, c("State", "male_pop", "female_pop")])

### Alternate (tidyr + stringr or base)
```r
us_census <- us_census %>%
  separate(GenderPop, c("male_pop", "female_pop"), "_") %>%
  mutate(male_pop = str_remove(male_pop, "M"),
         female_pop = str_remove(female_pop, "F"))
```

## 12. Convert cleaned columns to numeric

In [ ]:
num_cols <- c(race_cols, "Income", "male_pop", "female_pop")

us_census <- us_census %>%
  mutate(across(all_of(num_cols), as.numeric))

str(us_census)

## 13. Convert race percentages to decimal form (divide by 100)

In [ ]:
us_census <- us_census %>%
  mutate(across(all_of(race_cols), ~ .x / 100))

head(us_census[, c("State", race_cols)])

## 14–16. Detect and remove duplicate rows

In [ ]:
# 14. How many duplicates?
cat("Duplicate counts before distinct():\n")
print(us_census %>% duplicated() %>% table())

# 15. Keep only unique rows
us_census <- us_census %>% distinct()

# 16. Confirm
cat("\nDuplicate counts after distinct():\n")
print(us_census %>% duplicated() %>% table())

cat("\nFinal dimensions:", nrow(us_census), "rows x", ncol(us_census), "columns\n")

## 17. Final clean data frame — ready for analysis

In [ ]:
head(us_census)
summary(us_census$Income)
cat("\nMean % White:", mean(us_census$White) * 100, "\n")
cat("States / territories:", nrow(us_census), "\n")

# Optional: persist the clean artifact
# write_csv(us_census, "data/us_census_clean.csv")

---
## Alternate Code Paths (same result)

### One-liner race + Income cleaning with base R
```r
for (c in race_cols) us_census[[c]] <- as.numeric(gsub("%", "", us_census[[c]])) / 100
us_census$Income <- as.numeric(gsub("\\$", "", us_census$Income))
```

### GenderPop with strsplit (no tidyr)
```r
parts <- strsplit(as.character(us_census$GenderPop), "_")
us_census$male_pop   <- as.numeric(gsub("M", "", sapply(parts, `[`, 1)))
us_census$female_pop <- as.numeric(gsub("F", "", sapply(parts, `[`, 2)))
us_census$GenderPop  <- NULL
```

---
## More Practice

1. Create a new column `female_share = female_pop / (male_pop + female_pop)`. Which states have the highest female share?
2. Which state has the highest Hispanic percentage? The lowest White percentage?
3. Compute the correlation between `Income` and each race share. What is the strongest relationship?
4. Filter to states where TotalPop > 10 million and rank them by Income.
5. Reconstruct an approximate total population from male_pop + female_pop and compare with the given TotalPop (data-quality check).

In [ ]:
# Practice solutions (run after the main pipeline)
us_census <- us_census %>%
  mutate(female_share = female_pop / (male_pop + female_pop),
         pop_check    = male_pop + female_pop,
         pop_diff     = TotalPop - pop_check)

cat("Top 5 female share:\n")
print(us_census %>% arrange(desc(female_share)) %>% select(State, female_share) %>% head(5))

cat("\nHighest Hispanic:\n")
print(us_census %>% arrange(desc(Hispanic)) %>% select(State, Hispanic) %>% head(3))

cat("\nCorrelation Income ~ race shares:\n")
print(cor(us_census[, c("Income", race_cols)], use = "complete.obs")["Income", -1])

cat("\nLarge states by Income:\n")
print(us_census %>% filter(TotalPop > 1e7) %>% arrange(desc(Income)) %>% select(State, TotalPop, Income))

---
## Simulation Section

Modify a few parameters and observe how the final cleaned table changes.

**Parameters you can tweak:**
- `noise_pct` – random additive noise (as fraction) applied to Income before cleaning
- `drop_files` – number of CSV files to randomly omit (simulates missing extracts)
- `seed` – for reproducibility

In [ ]:
simulate_cleaning <- function(noise_pct = 0.0, drop_files = 0, seed = 42) {
  set.seed(seed)
  files <- list.files(path = "data", pattern = "states_.*\\.csv", full.names = TRUE)
  if (drop_files > 0 && drop_files < length(files)) {
    files <- sample(files, length(files) - drop_files)
  }
  df_list <- lapply(files, read_csv, show_col_types = FALSE)
  df <- bind_rows(df_list) %>% select(-1)

  # optional noise on the raw Income string is awkward; apply after numeric conversion
  race_cols <- c("Hispanic", "White", "Black", "Native", "Asian", "Pacific")
  df <- df %>%
    mutate(across(all_of(race_cols), ~ as.numeric(gsub("%", "", .x)) / 100),
           Income = as.numeric(gsub("\\$", "", Income)))

  if (noise_pct > 0) {
    df$Income <- df$Income * (1 + runif(nrow(df), -noise_pct, noise_pct))
  }

  df <- df %>%
    separate(GenderPop, c("male_pop", "female_pop"), "_") %>%
    mutate(male_pop = as.numeric(gsub("M", "", male_pop)),
           female_pop = as.numeric(gsub("F", "", female_pop))) %>%
    distinct()

  list(
    n_rows   = nrow(df),
    n_states = n_distinct(df$State),
    mean_income = mean(df$Income, na.rm = TRUE),
    mean_white  = mean(df$White, na.rm = TRUE) * 100
  )
}

# Baseline
cat("Baseline (no noise, all files):\n")
print(simulate_cleaning())

# With noise
cat("\nWith ±5% Income noise:\n")
print(simulate_cleaning(noise_pct = 0.05))

# Missing 2 files
cat("\nAfter randomly dropping 2 files:\n")
print(simulate_cleaning(drop_files = 2))

# Monte-Carlo: distribution of mean income when 1 file is missing
set.seed(123)
mc <- replicate(30, simulate_cleaning(drop_files = 1)$mean_income)
cat("\nMonte-Carlo mean Income (drop 1 file, 30 reps):\n")
print(summary(mc))

# Quick visual
hist(mc, main = "Simulated mean Income when 1 file missing", xlab = "Mean Income ($)", col = "#2E86AB", border = "white")

---
## Key Takeaways (Solution)

- Multi-file census extracts are cleaned once with a reproducible tidyverse pipeline.
- Symbols (%, $) and concatenated fields must be removed **before** `as.numeric()`.
- Overlapping files produce exact duplicates; always finish with `distinct()`.
- Scaling percentages to [0,1] makes downstream modelling cleaner.
- The same pattern (list → bind → clean strings → separate → type → distinct) works for any multi-file administrative data set.